# Proceedings

A set of tables for handling conference presentations and abstracts 

In [ ]:
#| default_exp Proceedings

In [ ]:
#| export
#| hide
from biobib.table import Table, tex_escape, colonify, to_int
from biobib.doc import Doc, NumbersDoc
from nbdev import show_doc
import pandas as pd

In [ ]:
#| hide
#| export
from fastcore.test import *
from fastcore.test import *
from fastcore.basics import *
from fastcore.foundation import *

### Author columns

Authors are found in columns of the proceedings spreadsheet that begin with `A`. The number after the `A` indicates the order of the author. For example, `A1` is the first author, `A2` is the second author, and so on. 

We set the `max_author` to the maximum number of authors we expect to see in a proceeding. This can vary across different proceedings lists. We use these max values to generate the columns for the authors.

In [ ]:
#| export
max_author=21
author_cols = ['A' + str(i+1) for i in range(max_author)]

#### Group affiliation symbols

Sometimes it is helpful to indicate group affiliations for authors. For example, if there are graduate students or undergraduate students who contributed to an article, we can indicate this with a symbol. We can define a dictionary of symbols and groups to use for this purpose. The dictionary keys are the symbols and the values are the groups. We can then use this dictionary to generate a column of symbols for the authors.

In [ ]:
#| export
from biobib.Publications import decorate_dict

## Define a template for conference presentations

Here is a template we can use for conference presentations. It generates a longtable which can span multiple pages, and provides the conference 

In [ ]:
#| export
proceedings_template = r'''% UC Bio-bib Proceedings Table
% Created on \VAR{created}

\BLOCK{for abstract in items} 
\VAR{abstract.Month|tex_escape}/\VAR{abstract.Year|tex_escape} & {\bf \VAR{abstract.Title|tex_escape}}. \VAR{abstract.Authors|tex_escape} \BLOCK{if abstract.Invited} (Invited) \BLOCK{endif} & \VAR{abstract.Conference|tex_escape}. (\VAR{abstract.Location|tex_escape}) \\
\BLOCK{endfor}

'''

### CV template 

This template formats proceedings into a similar reference style as publications. 

In [ ]:
#| export
proceedings_cv_template = r'''% UC CV Proceedings Table
% Created on \VAR{created}

\BLOCK{ set years = [] }
\BLOCK{ for proceeding in items if proceeding.Year not in years }
    \BLOCK{ do years.append(proceeding.Year) }
\BLOCK{ endfor }

\BLOCK{ if years }
    \BLOCK{ set max_year = years[-1] }
\BLOCK{ else }
    \BLOCK{ set max_year = None }
\BLOCK{ endif }


\mbox{\ \ \ \underline{\textbf{\VAR{max_year|int} }}}

\begin{etaremune}

\BLOCK{for proceeding in items}
\BLOCK{if (proceeding.Year == max_year) }
\VAR{proceeding|make_citation}
\BLOCK{ endif }
\BLOCK{ endfor }

\BLOCK{ for year in years|reverse }
\BLOCK{ if (year != max_year) }

\vspace{0.1in}
\mbox{\ \ \ \underline{\textbf{\VAR{year|int} }}}
\vspace{0.1in}

\BLOCK{for proceeding in items}
\BLOCK{if (proceeding.Year == year) }
\VAR{proceeding|make_citation}
\BLOCK{ endif }
\BLOCK{ endfor }
\BLOCK{ endif }
\BLOCK{ endfor }

\end{etaremune}
'''

In [ ]:
#| export
class Proceedings(Table):
    """
    A Proceedings class of Table for use in generating biobibs and CVs
    """
    def __init__(
            self, 
            doc:Doc=None, # Google sheet or Numbers object
            worksheet:str='Conference Abstracts', # worksheet name in google sheet
            csv_file:str=None, # optional csv file (if using csv files)
            cumulative:bool=False, # Is this table cumulative?
            ascending:bool=True, # Is this table sorted in ascending order?
            table_name:str='Proceedings', # table name for latex
            bold_author:bool=False, # bold the PI author?
            decorate_groups:bool=False, # decorate the PI author's group?
            template=proceedings_template # template content,
        ):
        filters = {
            'make_row': self.make_row,
            'make_citation': self.make_citation,
        }
        super().__init__(
            doc=doc,
            worksheet=worksheet,
            table_name=table_name,
            csv_file=csv_file, 
            template=template,
            filters=filters)
        self.cumulative = cumulative
        self.ascending = ascending
        self.bold_author = bold_author
        self.decorate_groups = decorate_groups
        self.df = self.clean_df()
        print(f'Initialized {self.table_name} table')
    
    @staticmethod
    def make_bold(fmt_cols, this_list):
        a = this_list.copy()
        for col in fmt_cols:
            a[col] = "-BOLD_START- " + a[col] + "-BOLD_END-"
        return a
    
    @staticmethod
    def add_decorator(fmt_cols, this_list, decorator="$^{*}$"):
        a = this_list.copy()
        for col in fmt_cols:
            a[col] = decorator+a[col]
        return a
    
    def _columns(self, column_string):
        try:
            return [int(a[1:])-1 for a in column_string.split(',')]
        except AttributeError:
            return []
    
    def make_author_list(self, row):
        # We're dealing with dict now, not a pd.Series...
        author_list = pd.Series(
            pd.Series(row)[author_cols].fillna('').values.tolist()
        )
        author_list = author_list[author_list.apply(len) > 0]
        return author_list.tolist()
    
    def format_authors(self, this_row):
        author_list = self.make_author_list(this_row)
        if self.bold_author:
            fmt_cols = self._columns(this_row['PI Author'])
            author_list = self.make_bold(fmt_cols, author_list)
        if self.decorate_groups:
            for column, decorator in decorate_dict.items():
                if this_row[column]:
                    if ','  in this_row[column]:
                        fmt_cols = self._columns(this_row[column].split(',')) 
                    else: 
                        fmt_cols = self._columns(this_row[column])
                    author_list = self.add_decorator(
                        fmt_cols, author_list, decorator=decorator)
        this_row['authors'] = ', '.join(author_list)
        return this_row
    
    def make_row(self, row):
        this_row = self.format_authors(this_row)
        assert 'authors' in this_row.keys()
        format_dict = {
            'year': tex_escape(to_int(this_row['Year'])),
            'month': tex_escape(to_int(this_row['Month'])),
            'title': tex_escape(this_row['Title'].strip()),
            'authors': tex_escape(this_row['authors']),
            # 'doi': self.doi(this_row['DOI']),
            # 'href': self.doi_link(this_row['DOI']),
            'conference': tex_escape(this_row['Conference']),
            'location': tex_escape(this_row['Location']),
        }

        format_string = "{month}/{year} & {title}. {authors} {conference} ({location})"
        row = format_string.format(**format_dict)
        row += "\\\\"
        return row

    def make_citation(self, this_row):
        try:
            this_row = self.format_authors(this_row)
        except Exception as e:
            print("Error formatting authors:")
            print(this_row)
            print(e)

        format_dict = {
            'year': tex_escape(to_int(this_row['Year'])),
            'month': tex_escape(this_row['Month']),
            'title': tex_escape(this_row['Title'].strip()),
            'authors': tex_escape(this_row['authors']),
            # 'doi': self.doi(this_row['DOI']),
            # 'href': self.doi_link(this_row['DOI']),
            'conference': tex_escape(this_row['Conference']),
            'location': tex_escape(this_row['Location']),
        }

        citation = "\item "

        format_string = "{authors} ({year}) {title}. {conference}, {location}."

        citation += format_string.format(**format_dict)
        return citation


## Provide a data cleaning function for the Proceedings table

In [ ]:
#| export
@patch
def clean_df(
        self:Proceedings,
        sort_by:list=['Year', 'Month'] # sort by column
        )->pd.DataFrame: # cleaned dataframe
    """
    Clean the Proceedings table.
    """
    print('Cleaning Proceedings table')
    df = Table.table_clean_df(self)
    # df['Topic'] = df['Title'] + ". " + df['Authors']
    df = df.sort_values(
        by=sort_by,
        ascending=[self.ascending]*len(sort_by)
    )
    # strip leading and trailing white space for all columns
    df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
    df.Year = df.Year.astype(int)
    df.Month = df.Month.astype(int)
    return df

In [ ]:
sheet_name = "CV.numbers"
doc = NumbersDoc(sheet_name)
doc.get_dataframe

<bound method NumbersDoc.get_dataframe of <biobib.doc.NumbersDoc object>>

In [ ]:
proceedings_table = Proceedings(
    doc=doc,
    table_name='Test',
    worksheet='Conference Abstracts',
    cumulative=True,
    ascending=False,
    template=proceedings_template)

Cleaning Proceedings table
Cleaning table for  Test
Initialized Test table


In [ ]:
test_eq(proceedings_table.table_name,'Test')
test_eq(list(proceedings_table.df.columns),[
    'Year', 'Month', '#', 'CODE', 'Authors', 
    'Undergrad Author', 'Visitor Author', 'PhD Committee Member', 
    'Graduate Advisee', 'Postdoctoral Advisee', 'PI Author',
     'A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 
     'A8', 'A9', 'A10', 'A11', 'A12', 'A13', 
     'A14', 'A15', 'A16', 'A17', 'A18', 'A19', 
     'A20', 'A21', 'Conference', 'Location', 
     'Invited', 'Organization', 'Pages', 'Title', 
     'Volume', 'Eval', 'Given?'
])

In [ ]:
proceedings_cv_table = Proceedings(
    doc=doc,
    table_name='Test_CV',
    worksheet='Conference Abstracts',
    cumulative=True,
    ascending=True, # The logic in the template assumes this.
    bold_author=True,
    template=proceedings_cv_template)

Cleaning Proceedings table
Cleaning table for  Test_CV
Initialized Test_CV table


In [ ]:
proceedings_cv_table.df.head()

,Year,Month,#,CODE,Authors,Undergrad Author,Visitor Author,PhD Committee Member,Graduate Advisee,Postdoctoral Advisee,...,A21,Conference,Location,Invited,Organization,Pages,Title,Volume,Eval,Given?
191,2001,5,C,C01C,"Caylor, K.K. and Shugart, HH",None,None,None,None,None,...,None,American Geophysical Union Fall Meeting,None,NaN,None,None,Variation in vegetation structure along a larg...,None,0.0,NaN
189,2001,12,A,C01A,"Shugart, HH and Caylor, K.K. and Dowty, PR",None,None,None,None,None,...,None,American Geophysical Union Fall Meeting,None,NaN,None,None,Ecological Scale Issues in Land/Atmosphere Cou...,None,0.0,NaN
190,2001,12,B,C01B,"Scanlon, TM and Albertson, JD and Caylor, K.K....",None,None,None,None,None,...,None,American Geophysical Union Fall Meeting,None,NaN,None,None,"Assessing Fractional Tree, Grass, and Bare Soi...",None,0.0,NaN
192,2001,12,D,C01D,"Caylor, K.K. and Dowty, PR and Shugart, HH",None,None,None,None,None,...,None,American Geophysical Union Fall Meeting,None,NaN,None,None,Relationship between small-scale structural he...,None,0.0,NaN
188,2002,5,A,C02A,"Caylor, K.K. and Shugart, HH",None,None,None,None,None,...,None,American Geophysical Union Fall Meeting,None,NaN,None,None,Simulated water-use of heterogeneous vegetatio...,None,0.0,NaN


In [ ]:
test_row = proceedings_cv_table.df.iloc[0].to_dict()
test_eq(proceedings_cv_table.make_author_list(test_row),['Caylor, K.K.', 'Shugart, H.H.'])

In [ ]:
proceedings_cv_table.write_template(path='../tex/CV/')

In [ ]:
print(proceedings_table.render_template())

% UC Bio-bib Proceedings Table
% Created on 2024-08-29 12:17

 
8/2024 & {\bf Co-regulation of canopy water use and temperature in desert trees}. Morgan, B., Trugman, A.T., and Caylor, K.  & Ecological Society of America. (Long Beach, CA) \\
 
8/2024 & {\bf Exploring rangeland zoogeochemistry at multiple scales to characterize animal impacts on ecosystem carbon}. Forbes, E., Young, H., Young, T. Schimel, J., Caylor, K.  & Ecological Society of America. (Long Beach, CA) \\
 
4/2024 & {\bf Fluxbot: The Next Generation - Design and Validation of a Wireless, Open-Source Mechatronic CO2 Flux Sensing Chamber}. Pan, C.; Patel, V.; Gewirtzman, J.; Richardson, I.; Dubey, R.; Caylor, K.; Dollar, A.; Forbes, E.  & ACM COMPASS 2024. (Delhi, India) \\
 
1/2024 & {\bf Global sensitivity of evapotranspiration to available soil moisture.}. Morgan, B., Araki, R., and Caylor, K.  & AGU Chapman Conference: Remote Sensing of the Water Cycle. (Honolulu, HI) \\
 
1/2024 & {\bf Sensitivity of dryland trees t

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()